In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import sklearn
sklearn_version = sklearn.__version__
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, log_loss,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)

RANDOM_STATE = 42


In [27]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

print('CWD:', Path.cwd())

ROOT = Path.cwd().resolve()
OUT_DIR = ROOT / 'out_data'
if not OUT_DIR.exists():
    hits = list(ROOT.rglob('out_data'))
    if hits:
        OUT_DIR = hits[0]

print('OUT_DIR:', OUT_DIR)

# --- load feature table produced earlier (110D + label y) ---
X_FILE = OUT_DIR / 'binary_vosa_XPcoeff_110d_L2.csv'
if not X_FILE.exists():
    raise FileNotFoundError(
        f'Missing {X_FILE}\n'
        'Expected the artifact created in the data-construction notebook.\n'
        'Make sure out_data/binary_vosa_XPcoeff_110d_L2.csv exists.'
    )

df_all = pd.read_csv(X_FILE)

feat_cols = [c for c in df_all.columns if re.fullmatch(r'c\d{3}', c)]

df_all['source_id'] = df_all['source_id'].astype('int64')
df_all['y'] = df_all['y'].astype(int)

X = df_all[feat_cols].to_numpy(dtype=np.float32)
y = df_all['y'].to_numpy(dtype=np.int64)
source_ids = df_all['source_id'].to_numpy(dtype=np.int64)

print('Loaded:', X_FILE.name)
print('Shape :', df_all.shape)
print('Positives:', int(df_all['y'].sum()), '| Negatives:', int((1 - df_all['y']).sum()))


CWD: /Users/kris/Desktop/Astrophysics
OUT_DIR: /Users/kris/Desktop/Astrophysics/out_data
Loaded: binary_vosa_XPcoeff_110d_L2.csv
Shape : (2815, 112)
Positives: 558 | Negatives: 2257


In [28]:
from sklearn.model_selection import train_test_split

# 70/15/15 (stratified)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp
)

print("Train:", X_train.shape, "pos_rate:", y_train.mean())
print("Val  :", X_val.shape,   "pos_rate:", y_val.mean())
print("Test :", X_test.shape,  "pos_rate:", y_test.mean())


Train: (1970, 110) pos_rate: 0.19847715736040608
Val  : (422, 110) pos_rate: 0.1966824644549763
Test : (423, 110) pos_rate: 0.19858156028368795


In [29]:
import numpy as np
from sklearn.metrics import confusion_matrix

def metrics_at_threshold(y_true, p_pos, thr: float):
    y_pred = (p_pos >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    precision   = tp / (tp + fp) if (tp + fp) else np.nan

    return float(sensitivity), float(specificity), float(precision), int(tn), int(fp), int(fn), int(tp)

def choose_threshold_youden(y_true, p_pos, grid_size=1001):
    thrs = np.linspace(0.0, 1.0, grid_size)
    best = None
    best_j = -np.inf

    for t in thrs:
        sens, spec, prec, tn, fp, fn, tp = metrics_at_threshold(y_true, p_pos, float(t))
        j = sens + spec - 1.0
        if j > best_j:
            best_j = j
            best = (float(t), sens, spec, prec, best_j)

    thr, sens, spec, prec, j = best
    return {"Threshold": thr, "Sensitivity": sens, "Specificity": spec, "Precision": prec, "YoudenJ": float(j)}

def choose_threshold_min_precision(y_true, p_pos, min_precision=0.90, grid_size=1001):
    # Find threshold with Precision >= min_precision; among them maximize Sensitivity.
    # If impossible, return threshold with max Precision.
    thrs = np.linspace(0.0, 1.0, grid_size)

    feasible = []
    best_prec = (-np.inf, None)

    for t in thrs:
        sens, spec, prec, *_ = metrics_at_threshold(y_true, p_pos, float(t))
        if prec > best_prec[0]:
            best_prec = (prec, (float(t), sens, spec, prec))
        if prec >= min_precision:
            feasible.append((float(t), sens, spec, prec))

    if feasible:
        thr, sens, spec, prec = max(feasible, key=lambda z: z[1])
        return {"Threshold": thr, "Sensitivity": sens, "Specificity": spec, "Precision": prec, "MinPrecision": float(min_precision), "Feasible": True}

    thr, sens, spec, prec = best_prec[1]
    return {"Threshold": thr, "Sensitivity": sens, "Specificity": spec, "Precision": prec, "MinPrecision": float(min_precision), "Feasible": False}


In [30]:
# --- Hyperparameter grid ---
# was: max C=100; now: max C=300
C_GRID = np.logspace(-2, 2, 21)

def pick_threshold_from_metric(y_true, p, metric="youden"):
    """
    Pick threshold based on either:
      - metric="youden": maximize (TPR - FPR)
      - metric="f1": maximize F1
    """
    thresholds = np.linspace(0.0, 1.0, 2001)  # fine grid for stability
    best_t = 0.5
    best_score = -np.inf

    y_true = np.asarray(y_true).astype(int)

    for t in thresholds:
        y_pred = (p >= t).astype(int)
        tp = np.sum((y_true == 1) & (y_pred == 1))
        tn = np.sum((y_true == 0) & (y_pred == 0))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))

        # avoid div0
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        if metric == "youden":
            score = tpr - fpr
        elif metric == "f1":
            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rec  = tpr
            score = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        else:
            raise ValueError("metric must be 'youden' or 'f1'")

        if score > best_score:
            best_score = score
            best_t = float(t)

    return best_t, float(best_score)

def basic_metrics_at_threshold(y_true, p, t):
    y_true = np.asarray(y_true).astype(int)
    y_hat = (np.asarray(p) >= t).astype(int)

    tp = np.sum((y_true == 1) & (y_hat == 1))
    tn = np.sum((y_true == 0) & (y_hat == 0))
    fp = np.sum((y_true == 0) & (y_hat == 1))
    fn = np.sum((y_true == 1) & (y_hat == 0))

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    return float(sens), float(spec), float(prec)

def prob_metrics(y_true, p):
    """
    Metrics that do NOT depend on a threshold.
    """
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)
    p_clip = np.clip(p, 1e-15, 1 - 1e-15)

    # guard: if some split accidentally ends up single-class (shouldn't happen, but keep robust)
    if len(np.unique(y_true)) < 2:
        return dict(roc_auc=np.nan, pr_auc=np.nan, brier=np.nan, logloss=np.nan)

    return dict(
        roc_auc=float(roc_auc_score(y_true, p)),
        pr_auc=float(average_precision_score(y_true, p)),
        brier=float(brier_score_loss(y_true, p)),
        logloss=float(log_loss(y_true, p_clip))
    )

def fit_predict_proba(model, X_tr, y_tr, X_eval):
    model.fit(X_tr, y_tr)
    return model.predict_proba(X_eval)[:, 1]

def tune_and_eval(selection_metric="youden"):
    """
    selection_metric:
      - "youden": choose (C, threshold) maximizing YoudenJ on VAL
      - "f1": choose (C, threshold) maximizing F1 on VAL

    Returns:
      df_val_thresholded, df_test_thresholded, df_test_probmetrics
    """
    variants = []

    # Existing variants (keep)
    variants.append(("L2", LogisticRegression(penalty="l2", solver="liblinear", max_iter=5000)))
    variants.append(("L2, balanced", LogisticRegression(penalty="l2", solver="liblinear", class_weight="balanced", max_iter=5000)))
    variants.append(("L1", LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000)))
    variants.append(("L1, balanced", LogisticRegression(penalty="l1", solver="liblinear", class_weight="balanced", max_iter=5000)))

    # Added earlier in your notebook (keep)
    variants.append(("L2 + zscore", Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(penalty="l2", solver="liblinear", max_iter=5000))])))
    variants.append(("L2 + zscore, balanced", Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(penalty="l2", solver="liblinear", class_weight="balanced", max_iter=5000))])))
    variants.append(("L1 + zscore", Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000))])))
    variants.append(("L1 + zscore, balanced", Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(penalty="l1", solver="liblinear", class_weight="balanced", max_iter=5000))])))

    # Calibrated-like variants you had (keep as-is: we assume you already created these earlier)
    if "make_calibrated_lr" in globals():
        variants.append(("L2 calibrated", make_calibrated_lr(penalty="l2", balanced=False)))
        variants.append(("L2 calibrated, balanced", make_calibrated_lr(penalty="l2", balanced=True)))

    best_rows_val = []
    best_rows_test = []
    best_rows_test_prob = []

    total_iters = len(variants) * len(C_GRID)
    with tqdm(total=total_iters, desc="Hyperparam search", unit="fit") as pbar:
        for name, base_model in variants:
            best = dict(score=-np.inf)

            for C in C_GRID:
                # clone-ish: rebuild per loop to avoid state carryover
                model = base_model
                # set C safely for pipeline or raw LR
                if isinstance(model, Pipeline):
                    model = Pipeline(model.steps)
                    model.named_steps['lr'].set_params(C=float(C))
                else:
                    # for non-pipeline non-calibrated LR
                    if hasattr(model, 'set_params') and 'C' in model.get_params():
                        model = LogisticRegression(**model.get_params())
                        model.set_params(C=float(C))

                # fit on TRAIN -> probs on VAL
                p_val = fit_predict_proba(model, X_train, y_train, X_val)

                # pick threshold on VAL
                t, sel_score = pick_threshold_from_metric(y_val, p_val, metric=selection_metric)
                pbar.update(1)

                if sel_score > best['score']:
                    best = dict(score=sel_score, C=float(C), thr=float(t), model=model)

            # Evaluate chosen (C, thr) on VAL (thresholded metrics)
            p_val_best = fit_predict_proba(best['model'], X_train, y_train, X_val)
            sens_v, spec_v, prec_v = basic_metrics_at_threshold(y_val, p_val_best, best['thr'])

            best_rows_val.append(dict(
                Variant=name, **{'Best C': best['C'], 'Threshold': best['thr']},
                **{'Sensitivity': sens_v, 'Specificity': spec_v, 'Precision': prec_v}
            ))

            # Evaluate chosen (C, thr) on TEST (thresholded metrics)
            p_test = fit_predict_proba(best['model'], X_train, y_train, X_test)
            sens_t, spec_t, prec_t = basic_metrics_at_threshold(y_test, p_test, best['thr'])

            best_rows_test.append(dict(
                Variant=name, **{'Best C': best['C'], 'Threshold': best['thr']},
                **{'Sensitivity': sens_t, 'Specificity': spec_t, 'Precision': prec_t}
            ))

            # TEST probability metrics (no threshold)
            pm = prob_metrics(y_test, p_test)
            best_rows_test_prob.append(dict(
                Variant=name, **{'Best C': best['C']},
                **{'ROC AUC': pm['roc_auc'], 'PR AUC': pm['pr_auc'], 'Brier': pm['brier'], 'Log loss': pm['logloss']}
            ))

    df_val  = pd.DataFrame(best_rows_val)
    df_test = pd.DataFrame(best_rows_test)
    df_prob = pd.DataFrame(best_rows_test_prob)

    return df_val, df_test, df_prob


# --- Run both selection schemes ---
_, df_test_y, df_prob_y = tune_and_eval(selection_metric="youden")
_, df_test_f, df_prob_f = tune_and_eval(selection_metric="f1")

# --- Benchmark comparison (edit these manually) ---
# Provide separate benchmarks for Youden vs F1 selections
BENCHMARK = {
    'Youden': {
        'Sensitivity': 0.878,
        'Specificity': 0.965,
        'Precision': 0.838,
    },
    'F1': {
        'Sensitivity': 0.818,
        'Specificity': 0.985,
        'Precision': 0.920,
    },
}

def delta_vs_benchmark(df, selection_label):
    metrics = ['Sensitivity', 'Specificity', 'Precision']
    base = BENCHMARK.get(selection_label, BENCHMARK.get('Youden', {m:0.0 for m in metrics}))
    deltas = df[['Variant'] + metrics].copy()
    for m in metrics:
        deltas[m] = deltas[m] - base[m]
    deltas.insert(0, 'Selection', selection_label)
    return deltas

# Show benchmark deltas FIRST (separate tables)
_delta_y = delta_vs_benchmark(df_test_y, 'Youden')
_delta_f = delta_vs_benchmark(df_test_f, 'F1')

max_abs_y = float(np.nanmax(np.abs(_delta_y[['Sensitivity','Specificity','Precision']].to_numpy())))
max_abs_f = float(np.nanmax(np.abs(_delta_f[['Sensitivity','Specificity','Precision']].to_numpy())))

print("\nΔ vs benchmark — Youden:")
display(
    _delta_y.style
    .format({'Sensitivity':'{:+.3f}', 'Specificity':'{:+.3f}', 'Precision':'{:+.3f}'})
    .background_gradient(cmap='BrBG', subset=['Sensitivity','Specificity','Precision'], vmin=-max_abs_y, vmax=max_abs_y)
)

print("\nΔ vs benchmark — F1:")
display(
    _delta_f.style
    .format({'Sensitivity':'{:+.3f}', 'Specificity':'{:+.3f}', 'Precision':'{:+.3f}'})
    .background_gradient(cmap='BrBG', subset=['Sensitivity','Specificity','Precision'], vmin=-max_abs_f, vmax=max_abs_f)
)

# --- Display: keep your existing TEST tables exactly as before ---
print('\nTEST — evaluated using VAL-picked (C, threshold) from Youden selection:')
display(
    df_test_y.style
    .format({'Best C':'{:.4g}', 'Threshold':'{:.3f}', 'Sensitivity':'{:.3f}', 'Specificity':'{:.3f}', 'Precision':'{:.3f}'})
    .background_gradient(cmap='viridis', subset=['Sensitivity','Specificity','Precision'])
)

print('\nTEST — evaluated using VAL-picked (C, threshold) from F1 selection:')
display(
    df_test_f.style
    .format({'Best C':'{:.4g}', 'Threshold':'{:.3f}', 'Sensitivity':'{:.3f}', 'Specificity':'{:.3f}', 'Precision':'{:.3f}'})
    .background_gradient(cmap='viridis', subset=['Sensitivity','Specificity','Precision'])
)

# --- probability-metric tables (TEST only), one per selection scheme ---
print('\nTEST — probability metrics (ROC AUC / PR AUC / Brier / Log loss) — Youden-picked model')
display(
    df_prob_y.style
    .format({'Best C':'{:.4g}', 'ROC AUC':'{:.4f}', 'PR AUC':'{:.4f}', 'Brier':'{:.5f}', 'Log loss':'{:.5f}'})
    .background_gradient(cmap='viridis', subset=['ROC AUC','PR AUC'])
    .background_gradient(cmap='viridis_r', subset=['Brier','Log loss'])
)



Hyperparam search: 100%|██████████| 168/168 [05:14<00:00,  1.87s/fit] 


Δ vs benchmark — Youden:


,Selection,Variant,Sensitivity,Specificity,Precision
0,Youden,L2,-0.092,-0.056,-0.158
1,Youden,"L2, balanced",-0.092,-0.042,-0.121
2,Youden,L1,-0.104,-0.012,-0.036
3,Youden,"L1, balanced",-0.116,-0.009,-0.028
4,Youden,L2 + zscore,-0.092,-0.024,-0.071
5,Youden,"L2 + zscore, balanced",-0.080,-0.039,-0.110
6,Youden,L1 + zscore,-0.068,-0.030,-0.082
7,Youden,"L1 + zscore, balanced",-0.080,-0.042,-0.118



Δ vs benchmark — F1:


,Selection,Variant,Sensitivity,Specificity,Precision
0,F1,L2,-0.056,-0.038,-0.140
1,F1,"L2, balanced",-0.068,-0.029,-0.112
2,F1,L1,-0.056,-0.020,-0.078
3,F1,"L1, balanced",-0.056,-0.026,-0.099
4,F1,L2 + zscore,-0.044,-0.038,-0.137
5,F1,"L2 + zscore, balanced",-0.020,-0.044,-0.150
6,F1,L1 + zscore,-0.044,-0.041,-0.146
7,F1,"L1 + zscore, balanced",-0.032,-0.041,-0.144


...

,Variant,Best C,Threshold,Sensitivity,Specificity,Precision
0,L2,39.81,0.123,0.786,0.909,0.680
1,"L2, balanced",63.1,0.338,0.786,0.923,0.717
2,L1,15.85,0.135,0.774,0.953,0.802
3,"L1, balanced",10,0.385,0.762,0.956,0.810
4,L2 + zscore,0.03981,0.252,0.786,0.941,0.767
5,"L2 + zscore, balanced",0.01585,0.397,0.798,0.926,0.728
6,L1 + zscore,0.2512,0.154,0.810,0.935,0.756
7,"L1 + zscore, balanced",0.1585,0.397,0.798,0.923,0.720



TEST — evaluated using VAL-picked (C, threshold) from F1 selection:


,Variant,Best C,Threshold,Sensitivity,Specificity,Precision
0,L2,63.1,0.189,0.762,0.947,0.780
1,"L2, balanced",100,0.503,0.750,0.956,0.808
2,L1,6.31,0.227,0.762,0.965,0.842
3,"L1, balanced",10,0.427,0.762,0.959,0.821
4,L2 + zscore,0.02512,0.339,0.774,0.947,0.783
5,"L2 + zscore, balanced",0.0631,0.485,0.798,0.941,0.770
6,L1 + zscore,0.0631,0.296,0.774,0.944,0.774
7,"L1 + zscore, balanced",0.01585,0.578,0.786,0.944,0.776



TEST — probability metrics (ROC AUC / PR AUC / Brier / Log loss) — Youden-picked model


,Variant,Best C,ROC AUC,PR AUC,Brier,Log loss
0,L2,39.81,0.8793,0.7855,0.08098,0.28613
1,"L2, balanced",63.1,0.8794,0.8061,0.08805,0.32989
2,L1,15.85,0.8803,0.8258,0.06588,0.25170
3,"L1, balanced",10,0.8836,0.8270,0.07509,0.29094
4,L2 + zscore,0.03981,0.8808,0.8179,0.06773,0.31994
5,"L2 + zscore, balanced",0.01585,0.8781,0.8071,0.08016,0.36265
6,L1 + zscore,0.2512,0.8808,0.8176,0.06788,0.32109
7,"L1 + zscore, balanced",0.1585,0.8785,0.8076,0.08022,0.36407
